# Maternal Health — Idaho BRFSS 2020–2024 (women 18–49)

Summary and comparative charts for four thematic groups:

- **Part A — Physical health:** 14+ days poor physical health, fair/poor health, current asthma, diabetes, obesity (BMI ≥ 30), hypertension
- **Part B — Mental health:** 14+ days poor mental health, depressive disorder, dissatisfaction with life, serious suicide ideation, unmet social/emotional support, social isolation, chronic stress
- **Part C — Health behaviors:** no routine checkup, no dental visit, no leisure-time physical activity, cigarette smoking, high alcohol consumption, past-30-day marijuana use
- **Part D — Social drivers of health:** forgone medical care due to cost, food insecurity, lost employment/hours, no primary insurance, insufficient transportation

Stratifiers for each group: health district, household income, education, race classification, binary residence (urban vs rural/frontier), BMI status.

**Data-quality conventions** (from the workbook's Notes sheet):

- Estimates with RSE > 30 are flagged as *CDC-unreliable* (light-grey hatched bars, `*` in hover).
- Rows with sample size < 50 are suppressed upstream.
- Error bars are 95% CIs.


In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_PATH = 'WomenPrecon_Full_AGG_Final4.3.26.xlsx'
SHEET = 'Precon Health, Aggr 2020-2024'
RSE_UNRELIABLE = 30

PHYS_METRICS = [
    ('14 or more Days of Poor Physical Health', '14+ Days Poor Physical Health'),
    ('Fair or Poor Health',                     'Fair or Poor Health'),
    ('Current Asthma',                          'Current Asthma'),
    ('Diagnosed with Diabetes',                 'Diabetes'),
    ('Obese (BMI >= 30)',                       'Obesity (BMI \u2265 30)'),
    ('High Blood Pressure',                     'Hypertension'),
]

MENTAL_METRICS = [
    ('14 or more Days of Poor Mental Health',             '14+ Days Poor Mental Health'),
    ('Diagnosed with Depressive Disorder',                'Depressive Disorder'),
    ('Always/Usually Feel Stressed',                      'Always/Usually Stressed'),
    ('Always/Usually Feel Socially Isolated',             'Always/Usually Isolated'),
    ('Seriously Considered Attempting Suicide',           'Serious Suicide Ideation'),
    ('Social/Emotional Support Needs Met Rarely/Never',   'Unmet Social/Emotional Support'),
    ('Dissatisfied/Very Dissatisfied with Life',          'Dissatisfied With Life'),
]

BEHAV_METRICS = [
    ('Did not have a Routine Checkup in the Last Year', 'No Routine Checkup (past year)'),
    ('Did not have Dental Visit in Past 12 months',     'No Dental Visit (past 12 mo)'),
    ('No Leisure Time Physical Activity',               'No Leisure-Time Physical Activity'),
    ('Current Cigarette Smoker',                        'Current Cigarette Smoker'),
    ('High Alcohol Consumption',                        'High Alcohol Consumption'),
    ('Marijuana Use in Past 30 Days',                   'Marijuana Use (past 30 days)'),
]

SDOH_METRICS = [
    ('Did Not Seek Medical Care Due to Cost',                  'Forgone Care (cost)'),
    ('Bought Food that Did Not Last (Always/Usually/Sometimes)', 'Food Insecurity'),
    ('Lost Employment/Hours Reduced',                          'Lost Employment / Hours'),
    ('No Primary Source of Insurance',                         'No Primary Insurance'),
    ('Insufficient Access to Transportation',                  'Insufficient Transportation'),
]

ALL_STATUSES = [s for s, _ in PHYS_METRICS + MENTAL_METRICS + BEHAV_METRICS + SDOH_METRICS]

DISTRICT_ORDER = [
    'Statewide',
    'Panhandle Public Health District',
    'North Central Public Health District',
    'Southwest Public Health District',
    'Central Public Health District',
    'South Central Public Health District',
    'Southeastern Idaho Public Health District',
    'Eastern Idaho Public Health District',
]
INCOME_ORDER = [
    'Less than $15k',
    '$15,000 - $24,999',
    '$25,000 - $34,999',
    '$35,000 - $49,999',
    '$50,000 - $74,999',
    '$75,000+',
]
INCOME_DISPLAY = {
    'Less than $15k':     '<$15k',
    '$15,000 - $24,999':  '$15-24k',
    '$25,000 - $34,999':  '$25-34k',
    '$35,000 - $49,999':  '$35-49k',
    '$50,000 - $74,999':  '$50-74k',
    '$75,000+':           '$75k+',
}
EDU_ORDER = [
    'K-11th Grade',
    '12th Grade or GED',
    'Some College',
    'College Graduate+',
]
RACE_ORDER = [
    'White, Non-Hispanic',
    'Hispanic',
    'Other, Non-Hispanic',
]
RESIDENCE_ORDER = [
    'Urban',
    'Rural/Frontier',
]
BMI_ORDER = [
    'Underweight or Normal Weight',
    'Overweight',
    'Obese',
]

BAR_COLOR = '#2e6fa7'
STATEWIDE_COLOR = '#08306b'
UNRELIABLE_COLOR = '#bfbfbf'
ERROR_COLOR = '#444'


## Load and prepare data

In [2]:
raw = pd.read_excel(DATA_PATH, sheet_name=SHEET)
for c in ['Percent', 'Lower_95CI', 'Upper_95CI', 'Sample_Size', 'Estimated_Adults', 'RSE']:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')

df = raw[raw['Status'].isin(ALL_STATUSES)].copy()
df['Unreliable'] = df['RSE'].fillna(0) > RSE_UNRELIABLE

print(f'Rows after metric filter: {len(df):,}')
print(f'Rows flagged RSE>{RSE_UNRELIABLE}: {df["Unreliable"].sum()}')
df.head()


Rows after metric filter: 7,872
Rows flagged RSE>30: 1359


,Years,Status,Category,Group,Region,Percent,Lower_95CI,Upper_95CI,Sample_Size,Estimated_Adults,RSE,Unreliable
0,2020:2024,14 or more Days of Poor Mental Health,College Attendance/Completion,Attended/Graduated College,Statewide,20.5,18.9,22.3,3910,47715.0,4.0,False
1,2020:2024,14 or more Days of Poor Mental Health,College Attendance/Completion,Did not Attend College,Statewide,28.7,26.0,31.4,1972,39767.0,5.0,False
2,2020:2024,14 or more Days of Poor Mental Health,College Attendance/Completion,Attended/Graduated College,Panhandle Public Health District,18.9,14.4,24.3,411,5188.0,13.0,False
3,2020:2024,14 or more Days of Poor Mental Health,College Attendance/Completion,Did not Attend College,Panhandle Public Health District,24.1,17.7,31.9,230,4391.0,15.0,False
4,2020:2024,14 or more Days of Poor Mental Health,College Attendance/Completion,Attended/Graduated College,North Central Public Health District,19.6,15.3,24.8,434,2458.0,12.0,False


## Chart helpers

In [3]:
def _hover(sub):
    out = []
    for _, r in sub.iterrows():
        if pd.notna(r['Lower_95CI']) and pd.notna(r['Upper_95CI']):
            ci = f"{r['Lower_95CI']:.1f}\u2013{r['Upper_95CI']:.1f}"
        else:
            ci = 'n/a'
        unrel = ' <b>*UNRELIABLE (RSE>30)*</b>' if r['Unreliable'] else ''
        n = int(r['Sample_Size']) if pd.notna(r['Sample_Size']) else 'n/a'
        rse = int(r['RSE']) if pd.notna(r['RSE']) else 'n/a'
        out.append(
            f"<b>{r['Group']}</b><br>"
            f"{r['Percent']:.1f}% (95% CI {ci})<br>"
            f"n = {n}, RSE = {rse}{unrel}"
        )
    return out


def _error_x(sub):
    return dict(
        type='data', symmetric=False,
        array=(sub['Upper_95CI'] - sub['Percent']).fillna(0).tolist(),
        arrayminus=(sub['Percent'] - sub['Lower_95CI']).fillna(0).tolist(),
        color=ERROR_COLOR, thickness=1.2, width=4,
    )


def _marker(sub, highlight_col=None, highlight_val=None):
    def pick(row):
        if row['Unreliable']:
            return UNRELIABLE_COLOR
        if highlight_col and row.get(highlight_col) == highlight_val:
            return STATEWIDE_COLOR
        return BAR_COLOR
    colors = [pick(r) for _, r in sub.iterrows()]
    patterns = ['/' if u else '' for u in sub['Unreliable']]
    return dict(
        color=colors,
        pattern=dict(shape=patterns, size=6, solidity=0.3),
        line=dict(color='#333', width=0.5),
    )


def _footer(fig, extra_notes='', y=-0.12):
    notes = (
        'Source: Idaho BRFSS 2020\u20132024, women ages 18\u201349. '
        'Error bars: 95% CI. '
        'Grey hatched bars: RSE > 30 (CDC-unreliable).'
    )
    if extra_notes:
        notes = f'{notes} {extra_notes}'
    fig.add_annotation(
        text=notes, xref='paper', yref='paper',
        x=0, y=y, xanchor='left', showarrow=False,
        font=dict(size=10, color='#555'),
    )


def statewide_summary(df, metrics, title):
    rows = []
    for status, label in metrics:
        sub = df[
            (df['Status'] == status)
            & (df['Category'] == 'Total')
            & (df['Region'] == 'Statewide')
        ]
        r = sub.iloc[0]
        rows.append({
            'Metric': label, 'Percent': r['Percent'],
            'Lower_95CI': r['Lower_95CI'], 'Upper_95CI': r['Upper_95CI'],
            'Sample_Size': r['Sample_Size'], 'RSE': r['RSE'],
            'Unreliable': bool(r['Unreliable']), 'Group': label,
        })
    summary = pd.DataFrame(rows).sort_values('Percent', ascending=True).reset_index(drop=True)
    fig = go.Figure(
        go.Bar(
            x=summary['Percent'], y=summary['Metric'], orientation='h',
            marker=_marker(summary), error_x=_error_x(summary),
            customdata=_hover(summary), hovertemplate='%{customdata}<extra></extra>',
        )
    )
    upper = summary['Upper_95CI'].max()
    x_max = max(upper if pd.notna(upper) else 0, summary['Percent'].max()) * 1.25
    for _, r in summary.iterrows():
        fig.add_annotation(
            x=r['Upper_95CI'] if pd.notna(r['Upper_95CI']) else r['Percent'],
            y=r['Metric'],
            text=f"  {r['Percent']:.1f}%",
            xanchor='left', yanchor='middle', showarrow=False,
            font=dict(size=12, color='#222'),
        )
    fig.update_layout(
        title=dict(text=title, x=0, xanchor='left'),
        template='plotly_white', height=90 + 55 * len(summary),
        margin=dict(l=280, r=100, t=70, b=110),
        xaxis=dict(title='% of women 18\u201349', ticksuffix='%', range=[0, x_max]),
        font=dict(family='Helvetica, Arial, sans-serif', size=12),
    )
    _footer(fig, y=-0.28)
    return fig, summary


def small_multiples(df, *, metrics, group_col, group_order, filters, title,
                    display_map=None, highlight_val=None, height=None,
                    footer_y=-0.12):
    n = len(metrics)
    rows = (n + 1) // 2
    cols = 2 if n > 1 else 1
    subtitles = [label for _, label in metrics]
    if n % 2 and cols == 2:
        subtitles = subtitles + ['']
    display_map = display_map or {}
    display_order = [display_map.get(g, g) for g in group_order]
    fig = make_subplots(
        rows=rows, cols=cols, subplot_titles=subtitles,
        horizontal_spacing=0.28, vertical_spacing=0.08,
    )
    for i, (status, _) in enumerate(metrics):
        row, col = i // cols + 1, i % cols + 1
        sub = df[df['Status'] == status].copy()
        for k, v in filters.items():
            sub = sub[sub[k] == v]
        sub = (
            sub.drop_duplicates(subset=[group_col])
               .set_index(group_col)
               .reindex(group_order)
               .reset_index()
        )
        sub['Unreliable'] = sub['Unreliable'].fillna(False)
        sub['_display'] = sub[group_col].map(lambda g: display_map.get(g, g))
        fig.add_trace(
            go.Bar(
                x=sub['Percent'], y=sub['_display'], orientation='h',
                marker=_marker(sub, group_col, highlight_val),
                error_x=_error_x(sub),
                customdata=_hover(sub), hovertemplate='%{customdata}<extra></extra>',
                showlegend=False,
            ),
            row=row, col=col,
        )
        upper = sub['Upper_95CI'].max()
        x_max = max(20, (upper if pd.notna(upper) else sub['Percent'].max()) * 1.2)
        fig.update_xaxes(
            ticksuffix='%', range=[0, x_max], gridcolor='#eee',
            row=row, col=col,
        )
        fig.update_yaxes(
            categoryorder='array', categoryarray=display_order,
            autorange='reversed', row=row, col=col,
        )
    if height is None:
        height = 200 + rows * 260
    highlight_note = ''
    if highlight_val:
        highlight_note = f'Dark-navy bar = {highlight_val} reference.'
    fig.update_layout(
        title=dict(text=title, x=0, xanchor='left'),
        height=height, template='plotly_white', bargap=0.25,
        margin=dict(l=220, r=40, t=90, b=140),
        font=dict(family='Helvetica, Arial, sans-serif', size=12),
    )
    _footer(fig, extra_notes=highlight_note, y=footer_y)
    return fig


---

# Part A — Physical health

Six metrics: poor physical health (≥14 days), fair/poor self-rated health, current asthma, diabetes, obesity (BMI ≥ 30), hypertension.

## A1. Statewide summary

In [4]:
phys_summary_fig, phys_summary = statewide_summary(
    df, PHYS_METRICS,
    'Statewide physical-health prevalence \u2014 women 18\u201349, Idaho 2020\u20132024',
)
phys_summary_fig.show()
phys_summary


,Metric,Percent,Lower_95CI,Upper_95CI,Sample_Size,RSE,Unreliable,Group
0,Diabetes,3.8,3.2,4.5,6006,9.0,False,Diabetes
1,14+ Days Poor Physical Health,10.9,9.9,12.1,5900,5.0,False,14+ Days Poor Physical Health
2,Hypertension,11.7,10.3,13.3,2727,7.0,False,Hypertension
3,Fair or Poor Health,11.9,10.9,13.0,6003,5.0,False,Fair or Poor Health
4,Current Asthma,12.7,11.6,13.8,5961,5.0,False,Current Asthma
5,Obesity (BMI ≥ 30),30.8,29.3,32.5,5280,3.0,False,Obesity (BMI ≥ 30)


## A2. By health district

Each panel = one metric across 7 districts, with Statewide as a darker reference bar.

In [5]:
small_multiples(
    df, metrics=PHYS_METRICS,
    group_col='Region', group_order=DISTRICT_ORDER,
    filters={'Category': 'Total'},
    highlight_val='Statewide',
    title='Physical health by health district \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## A3. By income

In [6]:
small_multiples(
    df, metrics=PHYS_METRICS,
    group_col='Group', group_order=INCOME_ORDER,
    filters={'Category': 'Income', 'Region': 'Statewide'},
    display_map=INCOME_DISPLAY,
    title='Physical health by household income \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## A4. By education

In [7]:
small_multiples(
    df, metrics=PHYS_METRICS,
    group_col='Group', group_order=EDU_ORDER,
    filters={'Category': 'Education', 'Region': 'Statewide'},
    title='Physical health by education \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## A5. By race classification

In [8]:
small_multiples(
    df, metrics=PHYS_METRICS,
    group_col='Group', group_order=RACE_ORDER,
    filters={'Category': 'Race Classification', 'Region': 'Statewide'},
    title='Physical health by race classification \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## A6. By binary residence (urban vs rural/frontier)

In [9]:
small_multiples(
    df, metrics=PHYS_METRICS,
    group_col='Group', group_order=RESIDENCE_ORDER,
    filters={'Category': 'Binary Residence Population Category', 'Region': 'Statewide'},
    title='Physical health by urban/rural residence \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## A7. By BMI status

Note: the *Obese (BMI ≥ 30)* metric stratified by the *Obese* BMI group is mechanically 100% (and 0% for the other groups) — read that panel as a consistency check, not as new signal.

In [10]:
small_multiples(
    df, metrics=PHYS_METRICS,
    group_col='Group', group_order=BMI_ORDER,
    filters={'Category': 'BMI Status', 'Region': 'Statewide'},
    title='Physical health by BMI status \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


---

# Part B — Mental health

Seven metrics: 14+ days poor mental health, diagnosed depressive disorder, chronic stress, social isolation, serious suicide ideation, unmet social/emotional support needs, dissatisfaction with life.

The lower-prevalence metrics (dissatisfaction with life, suicide ideation, unmet support) have higher sampling variability — expect more RSE>30 flags in the income and race breakdowns.

## B1. Statewide summary

In [11]:
mental_summary_fig, mental_summary = statewide_summary(
    df, MENTAL_METRICS,
    'Statewide mental-health prevalence \u2014 women 18\u201349, Idaho 2020\u20132024',
)
mental_summary_fig.show()
mental_summary


,Metric,Percent,Lower_95CI,Upper_95CI,Sample_Size,RSE,Unreliable,Group
0,Dissatisfied With Life,4.4,3.5,5.6,3097,11.0,False,Dissatisfied With Life
1,Unmet Social/Emotional Support,5.6,4.5,6.8,3116,10.0,False,Unmet Social/Emotional Support
2,Serious Suicide Ideation,7.7,6.1,9.7,1945,12.0,False,Serious Suicide Ideation
3,Always/Usually Isolated,9.0,7.8,10.5,3109,8.0,False,Always/Usually Isolated
4,Always/Usually Stressed,22.2,20.1,24.4,2387,5.0,False,Always/Usually Stressed
5,14+ Days Poor Mental Health,23.5,22.1,25.0,5896,3.0,False,14+ Days Poor Mental Health
6,Depressive Disorder,33.7,32.1,35.3,5980,2.0,False,Depressive Disorder


## B2. By health district

Each panel = one mental-health metric across 7 districts, with Statewide as a darker reference bar.

In [12]:
small_multiples(
    df, metrics=MENTAL_METRICS,
    group_col='Region', group_order=DISTRICT_ORDER,
    filters={'Category': 'Total'},
    highlight_val='Statewide',
    title='Mental health by health district \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## B3. By income

In [13]:
small_multiples(
    df, metrics=MENTAL_METRICS,
    group_col='Group', group_order=INCOME_ORDER,
    filters={'Category': 'Income', 'Region': 'Statewide'},
    display_map=INCOME_DISPLAY,
    title='Mental health by household income \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## B4. By education

In [14]:
small_multiples(
    df, metrics=MENTAL_METRICS,
    group_col='Group', group_order=EDU_ORDER,
    filters={'Category': 'Education', 'Region': 'Statewide'},
    title='Mental health by education \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## B5. By race classification

In [15]:
small_multiples(
    df, metrics=MENTAL_METRICS,
    group_col='Group', group_order=RACE_ORDER,
    filters={'Category': 'Race Classification', 'Region': 'Statewide'},
    title='Mental health by race classification \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## B6. By binary residence (urban vs rural/frontier)

In [16]:
small_multiples(
    df, metrics=MENTAL_METRICS,
    group_col='Group', group_order=RESIDENCE_ORDER,
    filters={'Category': 'Binary Residence Population Category', 'Region': 'Statewide'},
    title='Mental health by urban/rural residence \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## B7. By BMI status

In [17]:
small_multiples(
    df, metrics=MENTAL_METRICS,
    group_col='Group', group_order=BMI_ORDER,
    filters={'Category': 'BMI Status', 'Region': 'Statewide'},
    title='Mental health by BMI status \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


---

# Part C — Health behaviors

Six metrics: no routine checkup in the last year, no dental visit in the past 12 months, no leisure-time physical activity, current cigarette smoking, high alcohol consumption, marijuana use in the past 30 days.

## C1. Statewide summary

In [18]:
behav_summary_fig, behav_summary = statewide_summary(
    df, BEHAV_METRICS,
    'Statewide health-behavior prevalence \u2014 women 18\u201349, Idaho 2020\u20132024',
)
behav_summary_fig.show()
behav_summary


,Metric,Percent,Lower_95CI,Upper_95CI,Sample_Size,RSE,Unreliable,Group
0,Current Cigarette Smoker,10.6,9.7,11.6,5825,5.0,False,Current Cigarette Smoker
1,Marijuana Use (past 30 days),11.5,9.7,13.5,2218,8.0,False,Marijuana Use (past 30 days)
2,High Alcohol Consumption,16.9,15.6,18.3,5693,4.0,False,High Alcohol Consumption
3,No Leisure-Time Physical Activity,18.2,17.0,19.6,6004,4.0,False,No Leisure-Time Physical Activity
4,No Routine Checkup (past year),26.7,25.2,28.3,5880,3.0,False,No Routine Checkup (past year)
5,No Dental Visit (past 12 mo),29.3,27.3,31.4,3244,4.0,False,No Dental Visit (past 12 mo)


## C2. By health district

Each panel = one behavior across 7 districts, with Statewide as a darker reference bar.

In [19]:
small_multiples(
    df, metrics=BEHAV_METRICS,
    group_col='Region', group_order=DISTRICT_ORDER,
    filters={'Category': 'Total'},
    highlight_val='Statewide',
    title='Health behaviors by health district \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## C3. By income

In [20]:
small_multiples(
    df, metrics=BEHAV_METRICS,
    group_col='Group', group_order=INCOME_ORDER,
    filters={'Category': 'Income', 'Region': 'Statewide'},
    display_map=INCOME_DISPLAY,
    title='Health behaviors by household income \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## C4. By education

In [21]:
small_multiples(
    df, metrics=BEHAV_METRICS,
    group_col='Group', group_order=EDU_ORDER,
    filters={'Category': 'Education', 'Region': 'Statewide'},
    title='Health behaviors by education \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## C5. By race classification

In [22]:
small_multiples(
    df, metrics=BEHAV_METRICS,
    group_col='Group', group_order=RACE_ORDER,
    filters={'Category': 'Race Classification', 'Region': 'Statewide'},
    title='Health behaviors by race classification \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## C6. By binary residence (urban vs rural/frontier)

In [23]:
small_multiples(
    df, metrics=BEHAV_METRICS,
    group_col='Group', group_order=RESIDENCE_ORDER,
    filters={'Category': 'Binary Residence Population Category', 'Region': 'Statewide'},
    title='Health behaviors by urban/rural residence \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## C7. By BMI status

In [24]:
small_multiples(
    df, metrics=BEHAV_METRICS,
    group_col='Group', group_order=BMI_ORDER,
    filters={'Category': 'BMI Status', 'Region': 'Statewide'},
    title='Health behaviors by BMI status \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


---

# Part D — Social drivers of health

Five metrics: forgone medical care due to cost, food insecurity (household bought food that did not last always/usually/sometimes), lost employment or reduced hours, no primary source of insurance, insufficient access to transportation.

## D1. Statewide summary

In [25]:
sdoh_summary_fig, sdoh_summary = statewide_summary(
    df, SDOH_METRICS,
    'Statewide social-drivers-of-health prevalence \u2014 women 18\u201349, Idaho 2020\u20132024',
)
sdoh_summary_fig.show()
sdoh_summary


,Metric,Percent,Lower_95CI,Upper_95CI,Sample_Size,RSE,Unreliable,Group
0,Insufficient Transportation,9.4,8.2,10.9,3105,7.0,False,Insufficient Transportation
1,No Primary Insurance,10.7,9.6,11.8,4661,5.0,False,No Primary Insurance
2,Lost Employment / Hours,15.7,14.1,17.4,3108,5.0,False,Lost Employment / Hours
3,Forgone Care (cost),16.5,15.3,17.8,5993,4.0,False,Forgone Care (cost)
4,Food Insecurity,18.0,16.2,19.9,3103,5.0,False,Food Insecurity


## D2. By health district

Each panel = one driver across 7 districts, with Statewide as a darker reference bar.

In [26]:
small_multiples(
    df, metrics=SDOH_METRICS,
    group_col='Region', group_order=DISTRICT_ORDER,
    filters={'Category': 'Total'},
    highlight_val='Statewide',
    title='Social drivers of health by health district \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## D3. By income

In [27]:
small_multiples(
    df, metrics=SDOH_METRICS,
    group_col='Group', group_order=INCOME_ORDER,
    filters={'Category': 'Income', 'Region': 'Statewide'},
    display_map=INCOME_DISPLAY,
    title='Social drivers of health by household income \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## D4. By education

In [28]:
small_multiples(
    df, metrics=SDOH_METRICS,
    group_col='Group', group_order=EDU_ORDER,
    filters={'Category': 'Education', 'Region': 'Statewide'},
    title='Social drivers of health by education \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## D5. By race classification

In [29]:
small_multiples(
    df, metrics=SDOH_METRICS,
    group_col='Group', group_order=RACE_ORDER,
    filters={'Category': 'Race Classification', 'Region': 'Statewide'},
    title='Social drivers of health by race classification \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## D6. By binary residence (urban vs rural/frontier)

In [30]:
small_multiples(
    df, metrics=SDOH_METRICS,
    group_col='Group', group_order=RESIDENCE_ORDER,
    filters={'Category': 'Binary Residence Population Category', 'Region': 'Statewide'},
    title='Social drivers of health by urban/rural residence \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()


## D7. By BMI status

In [31]:
small_multiples(
    df, metrics=SDOH_METRICS,
    group_col='Group', group_order=BMI_ORDER,
    filters={'Category': 'BMI Status', 'Region': 'Statewide'},
    title='Social drivers of health by BMI status \u2014 women 18\u201349, Idaho 2020\u20132024',
).show()
